[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/examples/benchmark_noisy.ipynb)

# MISDA — noisy controlled benchmark

This notebook repeats the canonical controlled benchmark under a reproducible scale-relative observation-noise regime. Ground truth remains defined by the theoretical problem and the sampled clean objectives `Z`; MISDA receives only the noisy observation matrix `Y`.

The same Case 1–13 sequence used in `benchmark.ipynb` is retained so results can be compared directly. The case introductions are deliberately shorter and emphasize what the fixed noisy observation tests.


In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test the local code. In Colab, install main.
target = ".[benchmarks]" if Path("pyproject.toml").exists() else "git+https://github.com/monacofj/misda.git@main#egg=misda[benchmarks]"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import misda
import misda.benchmarks as bench

N = 300
SEED = 123
OBSERVATION_SEED = 456
SIGMA = 0.10


## Scale-relative observation noise

For each clean objective `Z_j`, observation follows `Y_j = Z_j + sigma * std(Z_j) * epsilon_j`, with `epsilon_j ~ N(0,1)`. Sampling and observation use distinct fixed seeds. Thus `sigma=0.10` means an observation-noise standard deviation equal to 10% of that clean objective's sample standard deviation.

This is a reproducible reference condition, not a claimed robustness threshold. Degradation across noise levels is studied separately in `diagnostic_robustness.ipynb`.


In [ ]:
noisy_results = {}

def run_case(problem_id):
    problem = bench.PROBLEM_BY_ID[problem_id]
    dataset = problem.generate(
        N=N,
        seed=SEED,
        sigma=SIGMA,
        observation_seed=OBSERVATION_SEED,
    )
    truth = bench.diagnostic_truth(problem, dataset.Z)
    mis_set = misda.discover(dataset.Y, name=truth["name"], seed=SEED)
    misda.evaluate(mis_set, metrics=("linear", "pareto"))
    benchmark_result = misda.benchmark(mis_set, truth)
    print(benchmark_result.report())
    mis_set.graph_plot()
    noisy_results[problem.id] = {
        "dataset": dataset,
        "result_obj": mis_set,
        "benchmark_obj": benchmark_result,
        "truth": truth,
    }
    return benchmark_result


## Case 1 - Independent objectives

**Reference structure.** Twenty independent generating variables are mapped directly to twenty objectives, `f_i = x_i`. Declared dimensions are `d_l=20`, `d_s=20`.

**Noise focus.** Noise tests whether weak spurious sample associations can create false redundancy among otherwise independent objectives.

**Reference expectation.** Under exact observation, MISDA should find no positive-redundancy structure and retain all twenty objectives.


In [ ]:
case_1 = run_case("independence")


## Case 2 - Complete positive redundancy

**Reference structure.** A single variable `x` is copied into all twenty objectives. Declared dimensions are `d_l=1`, `d_s=1`.

**Noise focus.** Noise tests whether the common one-dimensional signal remains strong enough for the redundant objectives to stay grouped.

**Reference expectation.** Under exact observation, MISDA should reduce the objective space to one representative.


In [ ]:
case_2 = run_case("total_redundancy")


## Case 3 - Four redundant blocks

**Reference structure.** Four independent factors generate four disjoint blocks; each factor is copied into five objectives. Declared dimensions are `d_l=4`, `d_s=4`.

**Noise focus.** Noise tests both within-block cohesion and resistance to spurious edges between the four independent blocks.

**Reference expectation.** Under exact observation, MISDA should retain one representative from each block, yielding dimension 4.


In [ ]:
case_3 = run_case("blocks_4x5")


## Case 4 - Two redundant blocks

**Reference structure.** Two independent factors each generate ten identical objectives. Declared dimensions are `d_l=2`, `d_s=2`.

**Noise focus.** Noise tests whether two large redundancy groups remain distinguishable after their exact within-block equality is perturbed.

**Reference expectation.** Under exact observation, MISDA should retain one representative from each block, yielding dimension 2.


In [ ]:
case_4 = run_case("blocks_2x10")


## Case 5 - Mixed independent and redundant objectives

**Reference structure.** Ten objectives are independently generated. Two additional independent factors are each replicated five times. Declared dimensions are `d_l=12`, `d_s=12`.

**Noise focus.** Noise tests whether MISDA can preserve isolated objectives while still recognizing the two noisy redundant groups.

**Reference expectation.** Under exact observation, MISDA should retain the ten independent objectives plus one representative from each redundant block.


In [ ]:
case_5 = run_case("mixed_independent_and_blocks")


## Case 6 - Nonlinear monotonic redundancy

**Reference structure.** One variable `x` generates twenty monotonic linear and nonlinear transforms. Declared dimensions are `d_l=1`, `d_s=1`.

**Noise focus.** Noise tests how stable the detected one-dimensional monotonic structure is when already non-identical transforms are additionally perturbed.

**Reference expectation.** Under exact observation, MISDA should recognize the common one-dimensional structure and select one representative.


In [ ]:
case_6 = run_case("monotonic_redundancy")


## Case 7 - Antagonistic linear groups

**Reference structure.** Ten objectives are copies of `x`; ten are copies of `-x`. Declared dimensions are `d_l=1`, `d_s=2`.

**Noise focus.** Noise tests whether the signed conflict remains identifiable while each positive-redundancy group is no longer composed of exact copies.

**Reference expectation.** Under exact observation, the signed dependence structure is one-dimensional, but the positive-redundancy structure requires one representative from each antagonistic group.


In [ ]:
case_7 = run_case("antagonistic_linear_groups")


## Case 8 - Trade-off with redundant families

**Reference structure.** Two generating variables `(a, b)` produce cost, consumption, and performance-related objective families of sizes 7, 7, and 6. Declared dimensions are `d_l=2`, `d_s=2`. Structural units are not uniquely declared for this case.

**Noise focus.** Noise tests whether the two-dimensional trade-off remains recoverable without turning the three observable families into an artificial structural count.

**Reference expectation.** Under exact observation, MISDA should recover the two-dimensional structure without treating the three generating families as three structural dimensions. Structural units are not uniquely declared for this case.


In [ ]:
case_8 = run_case("tradeoff_redundancies")


## Case 9 - Nonlinear redundant blocks

**Reference structure.** Four independent factors `(u, v, w, z)` each generate five nonlinear transforms. Declared dimensions are `d_l=4`, `d_s=4`.

**Noise focus.** Noise compounds nonlinear variation with observation error, testing whether the four generating blocks remain structurally separable.

**Reference expectation.** Under exact observation, MISDA should recover one representative per nonlinear block.


In [ ]:
case_9 = run_case("nonlinear_blocks_4x5")


## Case 10 - Antagonistic nonlinear groups

**Reference structure.** Ten monotonic transforms are generated from `x` and ten from the opposing quantity `1-x`. Declared dimensions are `d_l=1`, `d_s=2`.

**Noise focus.** Noise tests simultaneous stability of nonlinear within-group redundancy and antagonistic between-group dependence.

**Reference expectation.** Under exact observation, MISDA should preserve one representative from each antagonistic nonlinear group while recognizing their common latent degree of freedom.


In [ ]:
case_10 = run_case("antagonistic_nonlinear_groups")


## Case 11 - Overlapping latent factors

**Reference structure.** Two latent variables `(a, b)` generate objectives based on `a`, on `b`, and on the compound `a+b`, producing families of sizes 10, 4, and 6. Declared dimensions are `d_l=2`, `d_s=2`. Structural units are not uniquely declared for this case.

**Noise focus.** Noise tests whether the latent two-dimensional organization remains visible when the already overlapping observable families are perturbed.

**Reference expectation.** Under exact observation, MISDA should recover two-dimensional structure without forcing the overlapping families into an artificial two-block partition. Structural units are not uniquely declared for this case.


In [ ]:
case_11 = run_case("overlapping_factors")


# Adversarial diagnostics

The final two cases are already documented limitations under exact observation. In this notebook they remain adversarial references: the added noise is used to inspect the stability of the diagnostic evidence, not to redefine their ground truth or turn known mismatches into expected successes.


## Case 12 - Transitive positive chain

**Reference structure.** Twenty independent innovations generate a cumulative triangular chain: each successive objective contains the previous cumulative signal plus a new innovation. Declared dimensions are `d_l=20`, `d_s=20`.

**Noise focus.** Recovery is already expected to fail at `sigma=0`; under noise, the important question is whether the `TRANSITIVE_CHAINING` support diagnostic remains informative rather than whether the declared dimensions are recovered.

**Reference expectation.** The present graph construction collapses this chain to a much lower-dimensional description. The benchmark marks the resulting mismatch as the known `TRANSITIVE_CHAINING` limitation.


In [ ]:
case_12 = run_case("transitive_chain")


## Case 13 - Regime-switching dependence

**Reference structure.** Two variables `(a, b)` generate a smooth regime mixture `L(a,b)` and an additional `b` family; nonlinear transforms produce two groups of ten objectives. Declared dimensions are `d_l=2`, `d_s=2`.

**Noise focus.** Recovery is already expected to fail at `sigma=0`; noise tests whether the hidden-structure warning remains detectable as observation quality degrades.

**Reference expectation.** Pairwise graph evidence currently collapses the problem to one dimension. The benchmark retains this mismatch as the known `HIDDEN_SPECTRAL_STRUCTURE` limitation.


In [ ]:
case_13 = run_case("regime_switching")


# Suite summary

The table below gives the same cross-case summary used by the exact-observation benchmark, now for the fixed `sigma=0.10` reference condition.


In [ ]:
noisy_summary = misda.compile_benchmark_summary(noisy_results)
noisy_summary
